# Validation2.0: 54-Requirement Top-3 Recommendation Evaluation

This notebook evaluates the ontology-assisted online recommendation workflow on 54 requirements.

- Input: 19 enterprise requirements and 35 motivation requirements.
- Workflow: grouped prefiltering, merging all candidates scoring at least 65, close-reading every candidate, and then selecting the Top-3.
- Input source: all 54 cases come from the single canonical file `data/02_online_recommendation_workflow/01_input/01_user_requirements.csv`.
- Output: full run records are stored under workflow 02 `02_intermediate/`; Top-3 recommendations are stored under `03_output/`; evaluation scores are stored under `data/03_validation/03_output/01_recommendation_validation/`.
- Metrics: `Enterprise Requirements Candidate Top-3 Accuracy` and `Motivation nDCG@3`.
- GUI: the main GUI validation page reads and displays the saved results.


## 1. Setup

This section imports packaged functions only. It does not call the LLM and does not modify the ontology or knowledge graph.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'src':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import display

from src.opfmodelselection.user_requirements import USER_REQUIREMENTS
from src.opfmodelselection.validation2 import (
    VALIDATION2_RECOMMENDATION_INTERMEDIATE_DIR,
    VALIDATION2_RECOMMENDATION_OUTPUT_DIR,
    VALIDATION2_METRICS_OUTPUT_DIR,
    VALIDATION2_TOP3_CSV,
    VALIDATION2_RAW_JSON,
    RECOMMENDATION_HUMAN_GOLD_CSV,
    VALIDATION2_ACCURACY_BY_REQUIREMENT_CSV,
    VALIDATION2_ACCURACY_SUMMARY_CSV,
    evaluate_validation2_accuracy,
    load_validation2_top3_table,
    run_validation2_batch,
)

print('Canonical user-requirement count:', len(USER_REQUIREMENTS))
print('Recommendation intermediate directory:', VALIDATION2_RECOMMENDATION_INTERMEDIATE_DIR)
print('Recommendation output directory:', VALIDATION2_RECOMMENDATION_OUTPUT_DIR)
print('Validation metrics output directory:', VALIDATION2_METRICS_OUTPUT_DIR)


## 2. Preview the 54 Requirements

Run this section to check whether the 54 requirements are complete. It only displays the requirement list and does not call the LLM.


In [ ]:
requirements_df = pd.DataFrame(USER_REQUIREMENTS)
display(requirements_df[['requirement_id', 'requirement_group', 'requirement_name', 'requirement_text']])


## 3. Run Batch Recommendation for 54 Requirements

This section calls the LLM and runs the full online recommendation workflow for every requirement.

Output files

- `01_validation_batch_top3_recommendations.csv`: recommendation summary table
- `01_validation_batch_full_run_records.json`: complete intermediate results for audit

For a small test, pass a slice such as `USER_REQUIREMENTS[:2]`.


In [ ]:
# Full run: all 54 requirements. This may take a long time because it calls the LLM many times.

validation2_top3_df, validation2_raw_results = run_validation2_batch(
    requirements=USER_REQUIREMENTS,
    final_top_n=3,
    save=True,
    resume=True,          # Skip requirements that have already been completed.
    stop_on_error=True,   # Keep partial CSV/JSON results if an API quota or another error occurs.
)

print('Saved CSV:', VALIDATION2_TOP3_CSV)
print('Saved raw JSON:', VALIDATION2_RAW_JSON)
display(validation2_top3_df)


## 4. Load Saved Results

If section 3 has already been run, this section reloads the saved table. The GUI reads the same CSV.


In [ ]:
saved_validation2_df = load_validation2_top3_table()
if saved_validation2_df is None:
    print('No saved Validation2.0 result yet. Run section 3 first.')
else:
    print('Loaded saved result:', VALIDATION2_TOP3_CSV)
    display(saved_validation2_df)


## 5. Human Gold and Quantitative Evaluation

This section does not call the LLM again. It evaluates saved Top-3 recommendations against human gold answers.

- Enterprise requirements: `Enterprise Requirements Candidate Top-3 Accuracy`, the average share of Top-3 recommendations that belong to the human acceptable candidate set.
- Motivation requirements: ordered `nDCG@3`, with human relevance `Top1 = 4, Top2 = 0.5, Top3 = 0.5`.


In [ ]:
accuracy_table, summary_table, enterprise_gold_table, motivation_gold_table = evaluate_validation2_accuracy(save=True)

print('已保存量化评估汇总 / Saved quantitative evaluation summary:', VALIDATION2_ACCURACY_SUMMARY_CSV)
print('已保存逐需求评估表 / Saved per-requirement evaluation table:', VALIDATION2_ACCURACY_BY_REQUIREMENT_CSV)
print('54 条需求统一人工 Gold 来源 / Unified human Gold source for 54 requirements:', RECOMMENDATION_HUMAN_GOLD_CSV)

print('\n量化评估汇总 / Quantitative evaluation summary')
display(summary_table)
print('\n逐需求评估 / Evaluation by requirement')
display(accuracy_table)
print('\n企业需求人工候选集合 / Enterprise human candidate set')
display(enterprise_gold_table)
print('\nMotivation 人工有序 ranking / Motivation human ordered ranking')
display(motivation_gold_table)


## 6. Paper-Ingestion Quality Validation

This section treats 35 human-curated PaperModel records as the Gold Standard, faithfully translates their fields into English, and compares them with the 35 corresponding English LLM-ingested records. Each paper contributes 28 data-property judgments, for 980 field judgments in total.

Primary metrics:

- Overall average semantic similarity across the 35 papers
- Field accuracy

Evaluation rules:

- Each complete English Gold field is directly compared with the corresponding complete English LLM field. Long fields are not split, and feature fields are not converted into canonical labels.
- Semantic embeddings use `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`.
- `None`, `None reported`, `Not reported`, and equivalent values are treated as missing and skipped.
- Semantic similarity remains descriptive and is not used to decide factual accuracy.
- When both Gold and LLM values are present, the configured LLM API judges whether the extracted field is factually correct relative to the human Gold Standard.
- Paraphrasing, concise wording, and omission of optional detail are not penalized.
- If either side is missing, the field is skipped and excluded from the field-accuracy denominator.
- English Gold translations and LLM factual judgments are cached and reused unless their source values change.
- `hasModelName` is not graded, and Boolean `isScenarioSuitability` is excluded from sentence embeddings; therefore the evaluation contains exactly 28 data properties per paper.

The workflow no longer compares human and LLM records separately with PDF text, and it no longer performs a cross-language Gold-vs-LLM comparison.


In [ ]:
from src.opfmodelselection.semantic_evaluation import (
    PAPER_INGESTION_FACTUAL_ACCURACY_DETAILS_CSV,
    PAPER_INGESTION_SEMANTIC_SIMILARITY_DETAILS_CSV,
    PAPER_INGESTION_VALIDATION_SUMMARY_CSV,
    FIELD_FACTUAL_JUDGMENT_CACHE_JSON,
    HUMAN_GOLD_ENGLISH_TRANSLATION_CACHE_JSON,
    run_paper_ingestion_semantic_validation,
)

# Translate Gold fields, calculate descriptive similarity, and use the LLM for factual field judgments.
semantic_details_df, factual_details_df, overall_summary_df = run_paper_ingestion_semantic_validation(save=True)

print('Saved semantic details / 已保存语义相似度明细:', PAPER_INGESTION_SEMANTIC_SIMILARITY_DETAILS_CSV)
print('Saved factual details / 已保存事实准确率明细:', PAPER_INGESTION_FACTUAL_ACCURACY_DETAILS_CSV)
print('Saved overall summary / 已保存总体汇总:', PAPER_INGESTION_VALIDATION_SUMMARY_CSV)
print('Saved English Gold translation cache / 已保存人工Gold英文翻译缓存:', HUMAN_GOLD_ENGLISH_TRANSLATION_CACHE_JSON)
print('Saved LLM factual-judgment cache / 已保存LLM事实判断缓存:', FIELD_FACTUAL_JUDGMENT_CACHE_JSON)

display(overall_summary_df)
display(semantic_details_df)
display(factual_details_df)
